# Segmentazione RFM - Online Retail II

Notebook di segmentazione clienti tramite RFM (Recency, Frequency, Monetary), a partire dai dati puliti in `online_retail_clean.csv`.

**Nota**: questo notebook e' stato scritto per colmare un vuoto nel progetto - il file `data/processed/customer_rfm.csv` era gia' presente (generato da un processo esterno a questo repository) ma il notebook che lo produceva non era mai stato versionato. La metodologia qui sotto (quartili su R/F/M, regole di segmentazione) e' una ricostruzione ragionevole e standard, non necessariamente identica all'originale: i nomi dei 4 segmenti (`Top`, `Fedele`, `A rischio`, `Perso`) sono stati mantenuti per compatibilita' con i notebook a valle (`03_abbandono_stagionalita`, `05_clv`), ma le soglie esatte sono definite qui.

In [1]:
import pandas as pd

## 1. Caricamento dati puliti

In [2]:
df = pd.read_csv("../../data/processed/online_retail_clean.csv", parse_dates=["InvoiceDate"])
print(f"Transazioni: {len(df):,}")
df.head()

Transazioni: 779,423


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 2. Calcolo di Recency, Frequency, Monetary per cliente

- **Recency**: giorni trascorsi tra l'ultimo acquisto del cliente e una data di riferimento ("snapshot"), fissata al giorno successivo all'ultima transazione presente nel dataset.
- **Frequency**: numero di fatture (`Invoice`) distinte.
- **Monetary**: somma di `TotalPrice` (= `Quantity * Price`) su tutte le transazioni.

In [3]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Data di riferimento (snapshot): {snapshot_date}")

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda s: (snapshot_date - s.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalPrice", "sum"),
).reset_index()

print(f"Clienti: {len(rfm):,}")
rfm.describe()

Data di riferimento (snapshot): 2011-12-10 12:50:00


Clienti: 5,878


,CustomerID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.401327,6.289044,2914.112125
std,1715.572666,209.359711,13.009400,14245.372132
min,12346.000000,1.000000,1.000000,2.900000
25%,13833.250000,26.000000,1.000000,342.225000
50%,15314.500000,96.000000,3.000000,865.400000
75%,16797.750000,380.000000,7.000000,2247.170000
max,18287.000000,739.000000,398.000000,580987.040000


## 3. Punteggi RFM per quartili

Convertiamo ciascuna delle tre metriche in un punteggio da 1 (peggiore) a 4 (migliore) in base al quartile di appartenenza. Per `Recency` l'ordine e' invertito (meno giorni = punteggio piu' alto). Usiamo `rank(method="first")` prima di `qcut` per gestire in modo deterministico i molti valori ripetuti (es. `Frequency = 1` per centinaia di clienti), che altrimenti impedirebbero di dividere i dati in quartili di uguale dimensione.

In [4]:
rfm["R_score"] = pd.qcut(rfm["Recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)

rfm[["Recency", "R_score", "Frequency", "F_score", "Monetary", "M_score"]].head()

,Recency,R_score,Frequency,F_score,Monetary,M_score
0,529,1,11,4,372.86,2
1,2,4,8,4,4921.53,4
2,75,3,5,3,2019.40,3
3,19,4,4,3,4428.69,4
4,310,2,1,1,334.40,1


## 4. Regole di segmentazione

Applichiamo una regola in ordine di priorita' (il primo criterio soddisfatto assegna il segmento):

1. **Top**: punteggio alto su tutte e tre le dimensioni (`R>=3, F>=3, M>=3`) - clienti recenti, frequenti e di alto valore.
2. **A rischio**: punteggio di Recency basso (`R<=2`, non acquista da tempo) ma con frequenza o valore storicamente alti (`F>=3` o `M>=3`) - avevano valore, ora sono inattivi.
3. **Fedele**: non rientra nei casi precedenti, ma ha comunque frequenza o valore buoni (`F>=3` o `M>=3`) - tipicamente clienti recenti con un buon storico su una sola delle due dimensioni.
4. **Perso**: tutto il resto - punteggi bassi su frequenza e valore (spesso clienti con un solo acquisto), a prescindere dalla recency.

In [5]:
def assign_segment(row):
    if row["R_score"] >= 3 and row["F_score"] >= 3 and row["M_score"] >= 3:
        return "Top"
    if row["R_score"] <= 2 and (row["F_score"] >= 3 or row["M_score"] >= 3):
        return "A rischio"
    if row["F_score"] >= 3 or row["M_score"] >= 3:
        return "Fedele"
    return "Perso"


rfm["Segment"] = rfm.apply(assign_segment, axis=1)

## 5. Panoramica dei segmenti

In [6]:
print(rfm["Segment"].value_counts())
print()
rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].median()

Segment
Perso        2464
Top          1814
A rischio    1159
Fedele        441
Name: count, dtype: int64



,Recency,Frequency,Monetary
Segment,,,
A rischio,267.0,4.0,1313.670
Fedele,33.0,3.0,822.570
Perso,323.0,1.0,304.250
Top,22.0,9.0,3097.625


## 6. Salvataggio

Salviamo `CustomerID, Recency, Frequency, Monetary, Segment` in `data/processed/customer_rfm.csv`, mantenendo lo stesso formato gia' atteso dai notebook a valle (nessuna intestazione, separatore `;`, encoding con BOM UTF-8), per non dover modificare la logica di caricamento gia' scritta in `03_abbandono_stagionalita` e `05_clv`.

In [7]:
output_path = "../../data/processed/customer_rfm.csv"

rfm[["CustomerID", "Recency", "Frequency", "Monetary", "Segment"]].to_csv(
    output_path, sep=";", header=False, index=False, encoding="utf-8-sig"
)

print(f"Salvato in: {output_path}")
print(f"Righe: {len(rfm):,}")

Salvato in: ../../data/processed/customer_rfm.csv
Righe: 5,878
